### Package and Config Load


In [ ]:
# Imports
import os
from pathlib import Path

import napari
import numpy as np
import yaml
from bioio import BioImage

In [ ]:
# Imports from source
from spot_detector.config import PipelineConfig
from spot_detector.segmentation_detection import (
    assign_spots_to_mask,
    detect_spots_spotiflow,
    segment_2d,
    segment_3d,
)
from spot_detector.utils import ModelBundle

In [ ]:
# set cwd
project_root = Path.cwd().parent
os.chdir(project_root)

In [ ]:
config_path = config_path = Path("configs/config.yml")
if config_path.exists():
    with open(config_path) as f:
        config = PipelineConfig(**yaml.safe_load(f))
else:
    print("Config file not found at", config_path)

In [ ]:
data_folder = Path(config.paths.raw_data_dir).resolve()
file_list = [p for p in data_folder.iterdir() if p.is_file()]

##### Select Mode ---


In [ ]:
do_3d = config.mode.do_3d
mode = "3d" if do_3d else "2d"

##### Select File ---


In [ ]:
file_list

In [ ]:
file = file_list[0]

##### Select Scene ---


In [ ]:
img = BioImage(file)
img.scenes

In [ ]:
scene = 1

### Segmentation and Detection


In [ ]:
img.set_scene(scene_id=scene)

dim_order = "YX" if "Z" not in img.dims.order else "ZYX"
objects_stack = img.get_image_data(
    dim_order, C=config.channels.segmentation_image
).astype(np.float32)
spots_stack = img.get_image_data(dim_order, C=config.channels.spot_image).astype(
    np.float32
)

dx = img.physical_pixel_sizes.X
dz = img.physical_pixel_sizes.Z

In [ ]:
models = ModelBundle.load(config=config)

In [ ]:
print(f"    Segmenting ({mode})...")
if config.mode.do_3d:
    masks = segment_3d(
        bf_stack=objects_stack,
        model_cellpose=models.cellpose,
        factor=config.segmentation.bin_factor,
        stitch_threshold=config.segmentation.stitch_threshold,
    )
else:
    masks = segment_2d(
        bf_stack=objects_stack,
        model_cellpose=models.cellpose,
        factor=config.segmentation.bin_factor,
    )

n_obj = len(np.unique(masks)) - 1
print(f"    Found {n_obj} object(s) after border clearing")

print(f"    Detecting spots ({mode})...")
points, details = detect_spots_spotiflow(
    spot_stack=spots_stack,
    model_spotiflow=models.spotiflow,
    prob_thresh=config.detection.prob_thresh,
    min_distance=config.detection.min_distance,
    do_3d=config.mode.do_3d,
)
spot_labels = assign_spots_to_mask(coordinates=points, masks=masks)
print(f"    Detected {len(points)} spot(s), {(spot_labels > 0).sum()} assigned")

In [ ]:
print(f"    Detecting spots ({mode})...")
points, details = detect_spots_spotiflow(
    spot_stack=spots_stack,
    model_spotiflow=models.spotiflow,
    prob_thresh=config.detection.prob_thresh,
    min_distance=config.detection.min_distance,
    do_3d=config.mode.do_3d,
)

In [ ]:
getattr(details, "flow", None).head()

In [ ]:
print(
    f"details.ndim={getattr(details, 'flow', None).ndim}; details.shape={getattr(details, 'flow', None).shape}"
)

### Evaluation with Napari


In [ ]:
viewer = napari.Viewer()

if dz:
    scale = (dz, dx, dx)
else:
    scale = (dx, dx)

# object stack
viewer.add_image(
    objects_stack,
    name="Objects Image (BF)",
    colormap="gray",
    blending="additive",
    scale=scale,
)

# spot stack
viewer.add_image(
    spots_stack,
    name="Spots Image",
    colormap="magenta",
    blending="additive",
    scale=scale,
)

# masks
viewer.add_labels(masks, name="Cellpose Masks", opacity=0.4, scale=scale)

is_assigned = spot_labels > 0
point_colors = np.where(is_assigned, "green", "red")

# points
viewer.add_points(
    points,
    name="Spotiflow Spots",
    size=5,
    border_color=point_colors,
    face_color="transparent",
    border_width=0.1,
    properties={"assigned": is_assigned},
    scale=scale,
)